# Notebook para geração de pasta para o yousub - Modelo de integridade - Classificação

Autor: Viviane Sommer 

Entrega: 27/05/2025

## Objetivo:
Notebook para realizar inferência em um determinado vídeo e gerar a pasta consumida pela interface do YouSub.
Modelo de integridade.

## Instalações e Importações Necessárias

In [ ]:
import math
import os
import time
import json
from datetime import datetime, timedelta
import re
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.densenet import preprocess_input 
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import IoU
from tqdm import tqdm 
import subprocess
from utils.predict_functions import jaccard_distance
import boto3
from keras.utils import get_custom_objects
import mimetypes
import botocore.exceptions
import tempfile

## Declaração de Constantes 

#### PATH's dos vídeos e modelos

In [ ]:
bucket_name = 'analise-dados'
model_path = '../Modelo/modelcorrigido.h5'
classes_file_path = '../Modelo/training_classes.csv'
result_path = 'input_yousub'
video_keys = [
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-13-09-47-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-13-39-47-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-14-09-43-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-14-39-41-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-15-09-39-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-15-39-37-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-16-09-35-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-16-39-33-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-17-09-31-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-17-39-29-D030.mp4',
    'projeto-ia-submarina/ia-descomissionamento/operacoes-nordeste/108_SUB8ag24-108_6000685892 /3_VIDEOS/SUB8AG24-108-OS006000685892-2024-06-03-T-18-09-25-D030.mp4'
]
frame_skip = 150
predicoes_path=os.path.join(result_path, 'predicoes')
name_os = '108_SUB8ag24-108_6000685892'
local_folder_path = 'input_yousub'
s3_folder_path = 'projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/' + 'yousub_input_' + name_os

#### Configurar boto3 com credenciais, para acessar o S3.
 O arquivo deve-se chamar config.json e deve ter o seguinte formato:
 [
  {
    "aws_access_id": "SEU_AWS_ACCESS_KEY_ID",
    "secret_key": "SUA_AWS_SECRET_ACCESS_KEY"
  }
]

In [ ]:
with open("config.json",'r') as f:
    config = json.load(f)

data_config = config[0]
s3 = boto3.client( 's3', aws_access_key_id = data_config["aws_access_id"], aws_secret_access_key = data_config["secret_key"])

## Declaração de funções

### Funções específicas do YouSub

In [ ]:
def extract_timestamp_from_filename(filename):
    """
    Extracts a timestamp from a filename string in the format "YYYY-MM-DD-T-HH-MM-SS".

    This function uses a regular expression to search for a date and time pattern in the provided filename.
    If a match is found, it converts the extracted timestamp string into a `datetime` object.
    If no match is found, a `ValueError` is raised.

    Args:
        filename (str): The filename containing the timestamp in the format "YYYY-MM-DD-T-HH-MM-SS".

    Returns:
        datetime.datetime: A `datetime` object representing the extracted timestamp.

    Raises:
        ValueError: If no valid timestamp is found in the filename.
    """
    match = re.search(r"(\d{4}-\d{2}-\d{2})-T-(\d{2}-\d{2}-\d{2})", filename)
    if match:
        date_str = match.group(1)
        time_str = match.group(2)
        timestamp_str = f"{date_str} {time_str.replace('-', ':')}"
        timestamp_inicial = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S")
        return timestamp_inicial
    else:
        raise ValueError("Timestamp inicial não encontrado no nome do arquivo")


def calculate_final_timestamp(timestamp_inicial, duracao_segundos):
    """
    Calculates the final timestamp by adding a duration in seconds to an initial timestamp.

    This function takes an initial `datetime` object and adds a specified number of seconds to it,
    returning the resulting `datetime` object.

    Args:
        timestamp_inicial (datetime.datetime): The initial timestamp.
        duracao_segundos (int): The duration in seconds to be added to the initial timestamp.

    Returns:
        datetime.datetime: A `datetime` object representing the final timestamp after adding the duration.
    """
    timestamp_final = timestamp_inicial + timedelta(seconds=duracao_segundos)
    return timestamp_final


def get_video_duration(video_path):
    """
    Retrieves the duration of a video file in seconds using the `ffprobe` command-line utility.

    This function executes the `ffprobe` tool to extract the duration of the video stream from the given video file.
    The duration is rounded down to the nearest integer using `math.floor`.

    Args:
        video_path (str): The path to the video file whose duration is to be retrieved.

    Returns:
        int: The duration of the video in seconds.

    Raises:
        Exception: If `ffprobe` fails to execute or returns an error.
    """
    ffprobe_path = 'ffmpeg-7.0.2-amd64-static/ffprobe'
    cmd = [
        ffprobe_path, 
        '-v', 'error', 
        '-select_streams', 'v:0', 
        '-show_entries', 'stream=duration', 
        '-of', 'default=noprint_wrappers=1:nokey=1', 
        video_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise Exception(f"Error with ffprobe: {result.stderr}")
    return math.floor(float(result.stdout.strip()))


def build_sprite(miniatures, save_path, sprite_width=1920, sprite_height=1080, columns=10, lines=10):
    """
    Builds a sprite image by arranging a grid of miniature images and saves it to a specified path.

    This function creates a sprite by placing miniature images in a grid layout defined by the number of columns
    and lines. The dimensions of the sprite and the grid are specified by the `sprite_width`, `sprite_height`,
    `columns`, and `lines` arguments. The resulting sprite is saved to the given path.

    Args:
        miniatures (list of np.ndarray): A list of miniature images as NumPy arrays in the shape (height, width, 3).
        save_path (str): The file path where the resulting sprite image will be saved.
        sprite_width (int, optional): The width of the sprite image in pixels. Defaults to 1920.
        sprite_height (int, optional): The height of the sprite image in pixels. Defaults to 1080.
        columns (int, optional): The number of columns in the sprite grid. Defaults to 10.
        lines (int, optional): The number of lines (rows) in the sprite grid. Defaults to 10.

    Returns:
        None

    Raises:
        ValueError: If `miniatures` is empty or not a list of valid image arrays.
    """
    individual_width = sprite_width // columns
    individual_height = sprite_height // lines
    sprite = np.zeros((sprite_height, sprite_width, 3), dtype=np.uint8)
    for idx, miniature in enumerate(miniatures):
        if idx >= columns * lines:
            break
        if miniature.shape[1] != individual_width or miniature.shape[0] != individual_height:
            print(f"Image {miniature} has dimensions different from {individual_width}x{individual_height}")
            continue
        line = idx // columns
        col = idx % columns
        y_start = line * individual_height
        y_end = y_start + individual_height
        x_start = col * individual_width
        x_end = x_start + individual_width
        sprite[y_start:y_end, x_start:x_end] = miniature
    cv2.imwrite(save_path, sprite)
    print(f'Final sprite saved at {save_path}')


def extract_miniatures(video, num_miniatures=100, dims=(192, 108)):
    """
    Extracts a specified number of miniature frames from a video file.

    This function captures evenly spaced frames from the video, resizes them to the specified dimensions,
    and returns a list of these miniature frames.

    Args:
        video (str): The path to the video file.
        num_miniatures (int, optional): The number of miniature frames to extract. Defaults to 100.
        dims (tuple of int, optional): The dimensions (width, height) to resize each miniature frame. 
                                       Defaults to (192, 108).

    Returns:
        list of np.ndarray: A list of miniature frames as NumPy arrays, each resized to the specified dimensions.

    Raises:
        ValueError: If the video file cannot be opened or does not contain enough frames.
    """
    cap = cv2.VideoCapture(video)
    num_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    frame_skip = num_frames // num_miniatures
    miniatures = []
    for frame_count in range(num_miniatures):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_skip * frame_count)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, dims, interpolation=cv2.INTER_LINEAR)
        miniatures.append(frame)
    cap.release()
    return miniatures


def get_timestamp(file):
    """
    Extracts a timestamp from a file name in the format "YYYY-MM-DD-T-HH-MM-SS".
    This function uses a regular expression to search for a timestamp pattern in the provided file name.
    If the pattern matches, the timestamp is formatted and converted into a `datetime` object.
    If the pattern does not match or the formatting fails, the function returns `None` and prints an error message.

    Args:
        file (str): The file name containing the timestamp in the format "YYYY-MM-DD-T-HH-MM-SS".

    Returns:
        datetime.datetime or None: A `datetime` object representing the extracted timestamp, 
                                   or `None` if the timestamp could not be extracted or formatted.
    Raises:
        ValueError: If the timestamp formatting fails internally due to invalid date or time values.
    """
    pattern = r"(\d{4}-\d{2}-\d{2})-T-(\d{2})-(\d{2})-(\d{2})"
    format = "%Y-%m-%d %H:%M:%S"

    result = re.search(pattern, file)
    if result:
        try:
            date = result.group(1)
            hour = result.group(2)
            minute = result.group(3)
            second = result.group(4)

            formatted_timestamp = f"{date} {hour}:{minute}:{second}"
            return datetime.strptime(formatted_timestamp, format)
        except ValueError as e:
            print(f"Erro ao formatar o timestamp para o arquivo {file}: {e}")
            return None
    else:
        print(f"Padrão de timestamp não encontrado no nome do arquivo: {file}")

### Funções específicas do Modelo - Integridade

In [ ]:
def upload_folder_to_s3(local_folder_path, bucket_name, s3_folder_path, video_keys):
    """
    Uploads the contents of a local folder to an S3 bucket.
    This function walks through the local folder and uploads all files to the specified S3 bucket,
    preserving the folder structure by replicating the relative file paths in the S3 destination.

    Args:
        local_folder_path (str): The path to the local folder containing files to be uploaded.
        bucket_name (str): The name of the S3 bucket where files will be uploaded.
        s3_folder_path (str): The destination folder path in the S3 bucket.
        video_keys (list): A list of keys to be uploaded to S3.

    Returns:
        None

    Raises:
        Exception: If an error occurs during the folder traversal or file upload.
    """
    try:
        for root, dirs, files in os.walk(local_folder_path):
            for file in files:
                local_file_path = os.path.join(root, file)
                relative_path = os.path.relpath(local_file_path, local_folder_path)
                s3_file_key = os.path.join(s3_folder_path, relative_path)
                content_type, _ = mimetypes.guess_type(local_file_path)
                extra_args = {'ContentType': content_type} if content_type else {}
                try:
                    s3.upload_file(local_file_path, bucket_name, s3_file_key, ExtraArgs=extra_args)
                    print(f"Uploaded {local_file_path} to s3://{bucket_name}/{s3_file_key}")
                except Exception as upload_error:
                    print(f"Error uploading {local_file_path}: {upload_error}")
                    return

        print(f"Folder '{local_folder_path}' uploaded to s3://{bucket_name}/{s3_folder_path}")
    except Exception as e:
        print(f"Error walking through folder: {e}")


def upload_videos_to_s3(video_key, video, bucket_name):
    """
    Handles the upload of a video file to an S3 bucket after checking its existence.

    This function first checks if the specified video file exists in the S3 bucket.
    If it does not exist, the function terminates with a message. Otherwise, the video is downloaded
    to a temporary file locally, and then uploaded to the specified destination in the S3 bucket.

    Args:
        video_key (str): The key of the video in the S3 bucket to check and download.
        video (str): The local video file name to be uploaded to S3.
        bucket_name (str): The name of the S3 bucket where the video will be uploaded.

    Returns:
        None

    Raises:
        botocore.exceptions.ClientError: If there is an error checking the existence of the video in S3.
        Exception: If there is an error during the download or upload process.
    """
    try:
        s3.head_object(Bucket=bucket_name, Key=video_key)
    except botocore.exceptions.ClientError as e:
        if e.response['Error']['Code'] == '404':
            print(f"O vídeo {video_key} não existe no S3")
            return
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as temp_file:
            s3.download_fileobj(bucket_name, video_key, temp_file)
            video_path = temp_file.name
            s3_video_key = os.path.join(s3_folder_path, "videos", os.path.basename(video))
            content_type, _ = mimetypes.guess_type(video_path)
            extra_args = {'ContentType': content_type} if content_type else {}
            s3.upload_file(video_path, bucket_name, s3_video_key, ExtraArgs=extra_args)
            print(f"Uploaded video {video} to s3://{bucket_name}/{s3_video_key}")
            os.unlink(video_path) 
    except Exception as video_error:
        print(f"Error downloading/uploading video: {video_error}")


def load_model():
    """
    Loads a pre-trained TensorFlow model along with its class names.

    This function loads a TensorFlow/Keras model from a specified file path, using custom objects 
    such as `jaccard_distance`, `BinaryCrossentropy` loss, and `IoU` metric for model evaluation. 
    It also reads the class names from a separate file.

    Args:
        None

    Returns:
        tuple: A tuple containing:
            - model (tf.keras.Model): The loaded TensorFlow model.
            - class_names (list of str): A list of class names read from the classes file.

    Raises:
        FileNotFoundError: If the model file or classes file cannot be found.
        Exception: For errors during model loading or file reading.
    """
    get_custom_objects().update({'jaccard_distance': jaccard_distance})
    model = tf.keras.models.load_model(model_path, custom_objects={
        'loss': BinaryCrossentropy,
        'iou_score': IoU(num_classes=35, target_class_ids=[1] * 35)
    })
    with open(classes_file_path, 'r') as f:
        class_names = f.readline().strip().split(',')
    return model, class_names


def process_video(video, total_videos, columns, total_inferences, total_frames, frame_skip):
    """
    Processes a video file using a pre-trained model to perform frame-by-frame predictions.

    This function reads a video, extracts frames at specified intervals, preprocesses them, 
    and feeds them into a pre-trained model for inference. The results, along with timestamp and 
    frame information, are saved in a list.

    Args:
        video (str): The path to the video file to be processed.
        total_videos (int): The total number of videos being processed (used for tracking progress).
        columns (list): A list of column names for the output data (e.g., frame data and predictions).
        total_inferences (int): A counter for the total number of inferences performed (used for tracking progress).
        total_frames (int): The total number of frames in the video (used for tracking progress).
        frame_skip (int): The number of frames to skip between predictions.

    Returns:
        list: A list of rows, where each row contains:
            - Video name
            - Frame index
            - Time elapsed in seconds
            - Timestamp
            - Predictions for each class (formatted as strings with three decimal places)

    Raises:
        Exception: For errors related to file handling, video processing, model inference, or frame preprocessing.
    """
    model, class_list = load_model()
    if not model:
        print("Erro: Modelo não foi carregado corretamente.")
        return []

    df_inference = []
    try:
        model_shape = model.input_shape
        if isinstance(model_shape, list):
            model_shape = model_shape[0]
        print(f"Model input shape: {model_shape}")
    except AttributeError as e:
        print(f"Erro ao obter a forma do modelo: {e}")
        return []

    if not os.path.exists(video):
        print(f"Erro: Arquivo de vídeo não encontrado - {video}")
        return []

    cap = cv2.VideoCapture(video)
    if not cap.isOpened():
        print(f"Erro: Não foi possível abrir o vídeo - {video}")
        return []

    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    num_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    video_name = os.path.basename(video).split('.')[0]

    initial_timestamp = get_timestamp(video_name)
    if initial_timestamp is None:
        initial_timestamp = datetime.now() 
        print(f"Timestamp inicial não encontrado para {video_name}. Usando o timestamp atual: {initial_timestamp}")

    with tqdm(total=int(num_frames // frame_skip), desc=f"Processing {video_name}") as pbar:
        for prediction_index in range(0, int(num_frames), frame_skip):
            cap.set(cv2.CAP_PROP_POS_FRAMES, prediction_index)
            ret, frame = cap.read()

            if not ret:
                print(f"Erro ao ler o frame no índice {prediction_index}. Interrompendo o processamento.")
                break

            try:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (model_shape[1], model_shape[2]), interpolation=cv2.INTER_LINEAR)
                frame = preprocess_input(frame)
                frame = np.expand_dims(frame, 0)
            except Exception as e:
                print(f"Erro ao processar o frame no índice {prediction_index}: {e}")
                continue

            try:
                predictions = model.predict(frame, verbose=0)
            except Exception as e:
                print(f"Erro ao realizar a predição no índice {prediction_index}: {e}")
                continue

            col_tempo = int(prediction_index / np.ceil(frame_rate))
            current_timestamp = initial_timestamp + timedelta(seconds=col_tempo)
            col_timestamp = current_timestamp.strftime('%Y-%m-%d %H:%M:%S')

            for _, prediction in enumerate(predictions):
                prediction_formatted = ['%.3f' % round(elem, 3) for elem in prediction.tolist()]
                new_row = [video_name, prediction_index, col_tempo, col_timestamp] + prediction_formatted
                df_inference.append(new_row)

            total_inferences += 1
            pbar.update(1)

    cap.release()
    return df_inference


def run_inference(videos, columns, save_path, frame_skip=30):
    """
    Runs inference on a list of videos and saves the results to a CSV file.
    This function processes a list of video files by running frame-by-frame inference using a pre-trained model.
    The results, including predictions and timestamps, are saved into a Pandas DataFrame and exported to a CSV file.

    Args:
        videos (list of str): A list of paths to video files to be processed.
        columns (list of str): Column names for the output DataFrame.
        save_path (str): The directory where the resulting CSV file will be saved.
        frame_skip (int, optional): The number of frames to skip between predictions. Defaults to 30.

    Returns:
        None

    Raises:
        Exception: For errors related to video processing, file handling, or inference.
    """
    total_videos = len(videos)
    total_frames = sum([int(cv2.VideoCapture(video).get(cv2.CAP_PROP_FRAME_COUNT)) for video in videos]) // 30
    total_inferences = 0
    start_total_time = time.time()

    df_inference = []
    for video in videos:
        result = process_video(video, total_videos, columns, total_inferences, total_frames, frame_skip)
        df_inference.extend(result)

    end_total_time = time.time()
    total_processing_time = end_total_time - start_total_time
    minutes = int(total_processing_time / 60)
    seconds = total_processing_time % 60
    print(f'Total processing time: {minutes} minutes {seconds:.2f} seconds')

    df_inference = pd.DataFrame(df_inference, columns=columns)
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    df_inference.to_csv(os.path.join(save_path, 'predicoes_1s.csv'), index=False)


def download_video_from_s3(bucket_name, video_key, download_dir="downloads"):
    """
    Downloads a video file from an S3 bucket to a local directory.
    This function retrieves a video file from the specified S3 bucket and saves it to a local directory.
    If the directory does not exist, it is created automatically.

    Args:
        bucket_name (str): The name of the S3 bucket where the video is stored.
        video_key (str): The key (path) of the video file in the S3 bucket.
        download_dir (str, optional): The local directory where the video will be downloaded.
                                      Defaults to "downloads".

    Returns:
        str or None: The full local path to the downloaded video if successful, or `None` if an error occurs.

    Raises:
        Exception: If there is an error during the download process or while creating the local directory.
    """
    try:
        if not os.path.exists(download_dir):
            os.makedirs(download_dir)
        file_name = os.path.basename(video_key)
        download_path = os.path.join(download_dir, file_name)
        with open(download_path, "wb") as file:
            s3.download_fileobj(bucket_name, video_key, file)

        print(f"Vídeo baixado com sucesso: {download_path}")
        return download_path
    except Exception as e:
        print(f"Erro ao baixar o vídeo {video_key}: {e}")
        return None

## Download dos vídeos - S3

In [ ]:
videos = []
for video in tqdm(video_keys, desc="Baixando vídeos", unit="arquivo"):
    videos.append(download_video_from_s3(bucket_name, video))

## Inferência dos vídeos

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
    except RuntimeError as e:
        print(e)

_, class_list = load_model()
columns = ['Video', 'Frame', 'Tempo', 'Timestamp'] + class_list
run_inference(videos, columns, predicoes_path, frame_skip)

## Simulando geração dos dados de indexação do YouSub

O arquivo "prediction_index.json" é utilizado para mapear os nomes dos arquivos de predições. Atualmente, ele importa apenas os campos presentes em "predicoes.originais", que deve conter o nome do arquivo CSV, conforme a estrutura estabelecida.

Os campos "suavizada" e "agrupadas" foram projetados para complementar a visualização do gráfico, mas ainda não foram implementados. Portanto, é possível deixá-los vazios, sem afetar o funcionamento atual do sistema.

In [ ]:
data = {
    "predicoes": {
        "originais": {
            "periodo": 1,
            "arquivo": "predicoes_1s.csv"
        },
        "suavizadas": {
            "periodo": 1,
            "arquivo": ""
        },
        "agrupadas": {
            "periodo": 1,
            "arquivo": ""
        }
    }
}

json_file_path = os.path.join(result_path, 'prediction_index.json')
with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file, indent=4)
print(f"JSON file created at: {json_file_path}")

O arquivo pacote.json é gerado com os dados de identificação do pacote, incluindo pacoteHashId e nomePacote. No entanto, apenas o nomePacote é utilizado para alterar o título da página, enquanto o pacoteHashId não é utilizado.

In [ ]:
data = {
  "pacoteHashId": "zBLJpJS6zAY",
  "nomePacote": "INSP. PROG. - SIST. ANM PI-ANM 8-RO-52HP-RJS/8-RO-52HPA-RJS / 8-RO-52WA-RJS / P-54"
}
json_file_path = os.path.join(result_path, 'pacote.json')
with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file, indent=4)

print(f"JSON file created at: {json_file_path}")

Processamento da lista de vídeos e criar um arquivo JSON que contém informações sobre cada vídeo, chamado 'video_data.json'. 
As informações incluem o hash ID do vídeo, nome do arquivo, duração em segundos, caminho do sprite e timestamps inicial e final.

O processo é realizado da seguinte forma:

- O código itera sobre a lista de vídeos e, para cada vídeo, extrai o timestamp inicial, obtém a duração do vídeo e calcula o timestamp final.
- Em seguida, define o caminho do sprite e adiciona as informações do vídeo a uma lista de vídeos.
- O código também gera um sprite para cada vídeo e o salva em um diretório específico.
- Por fim, o código escreve as informações dos vídeos em um arquivo JSON chamado video_data.json, que é criado no diretório especificado.

O arquivo JSON resultante contém uma lista de objetos, cada um representando um vídeo, com as seguintes propriedades: hashid, nome, duracaoSegundos, sprite e timestamps.

In [ ]:
json_file_path = os.path.join(result_path, 'video_data.json')
videos_info = {"videos": []}
index = 1

for video in videos:
    hashid = index
    timestamp_inicial = extract_timestamp_from_filename(os.path.basename(video))
    duracao_segundos = get_video_duration(video)
    timestamp_final = calculate_final_timestamp(timestamp_inicial, duracao_segundos)
    sprite_folder = os.path.join(result_path, 'sprites')
    sprite_path = os.path.join(sprite_folder, f'sprite_{index}.jpg')
    video_info = {
        "hashid": f'hashid_do_video_{str(hashid)}',
        "nome": os.path.basename(video),
        "duracaoSegundos": duracao_segundos,
        "sprite": f'sprite_{index}.jpg',
        "timestamps": {
            "inicial": str(timestamp_inicial),
            "final": str(timestamp_final)
        }
    }
    videos_info["videos"].append(video_info)
    miniatures = extract_miniatures(video)
    build_sprite(miniatures, sprite_path)
    index += 1
with open(json_file_path, 'w') as json_file:
    json.dump(videos_info, json_file, indent=4)
print(f"JSON file created at: {json_file_path}")

## Aplicando suavização nas predições, com média móvel (Janela de tamanho 3 e 6)

In [ ]:
df = pd.read_csv('input_yousub/predicoes/predicoes_1s.csv')
columns_to_average = df.columns[4:]
window_sizes = [3, 6]
for window_size in window_sizes:
    for column in columns_to_average:
        df[f'{column}_moving_avg_{window_size}'] = df[column].rolling(window=window_size).mean()
output_directory = 'input_yousub/predicoes/smoothed'
os.makedirs(output_directory, exist_ok=True)
output_file_path = os.path.join(output_directory, 'predicoes_1s.csv')
df.to_csv(output_file_path, index=False)

## Enviando pasta input_yousub novamente para o S3

In [ ]:
upload_folder_to_s3(local_folder_path, bucket_name, s3_folder_path, video_keys)
for video_key, video in zip(video_keys, videos):
    upload_videos_to_s3(video_key, video, bucket_name)